# Start

In [ ]:
!pip install xgboost gensim tqdm hmmlearn pgmpy sklearn-crfsuite transformers tensorflow keras keras-tuner


# Import

In [ ]:
# Standard Library Imports
import os
import sys
import pickle
import joblib
import logging
import torch
import numpy as np
import pandas as pd
from statistics import mean

# Visualization Libraries
import matplotlib.pyplot as plt

# Machine Learning Libraries
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.preprocessing import MaxAbsScaler, MinMaxScaler, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, log_loss, hinge_loss
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, ClassifierMixin

from pgmpy.models import BayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

import nltk
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import defaultdict, Counter
from hmmlearn import hmm

nltk.download('punkt')
nltk.download('stopwords')

# Specialized Libraries
import xgboost as xgb
import hmmlearn.hmm
from hmmlearn.hmm import GaussianHMM
from sklearn_crfsuite import CRF

# Natural Language Processing (NLP) Libraries
import gensim.downloader as api
from transformers import AutoTokenizer, AutoModel

# Progress Bar
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tqdm import tqdm

import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from keras_tuner import RandomSearch


In [ ]:
## Options
pd.set_option("max_colwidth", None)

# Logger

In [ ]:
class MyLogger:
    """
        Initializes the MyLogger with a specified log file for logging messages.
        
        :param log_file: str, path to the log file (default: 'app.log')
    """
    def __init__(self, log_file='app.log'):
        self.log_file = log_file
        self._initialize_logger()

    def _initialize_logger(self):
        """
        Initializes the MyLogger with a specified log file for logging messages.
        
        :param log_file: str, path to the log file (default: 'app.log')
        """
        if os.path.exists(self.log_file):
            file_mode = 'a'
        else:
            file_mode = 'w'

        self.logger = logging.getLogger()
        self.logger.setLevel(logging.INFO)

        file_handler = logging.FileHandler(self.log_file, mode=file_mode, encoding='utf-8')
        file_handler.setLevel(logging.INFO)

        console_handler = logging.StreamHandler()
        console_handler.setLevel(logging.INFO)

        formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
        file_handler.setFormatter(formatter)
        console_handler.setFormatter(formatter)

        if self.logger.hasHandlers():
            self.logger.handlers.clear()

        self.logger.addHandler(file_handler)
        self.logger.addHandler(console_handler)

    def log_message(self, message):
        """
        Logs an info-level message to the current log file and console.
        
        :param message: str, the message to be logged
        """
        self.logger.info(message)

    def change_log_file(self, new_log_file):
        """
        Changes the log file path and reinitializes the logger with the new file.
        
        :param new_log_file: str, the new path to the log file
        """
        self.log_file = new_log_file
        self._initialize_logger()

logger = MyLogger()

os.environ["TOKENIZERS_PARALLELISM"] = "true"

# Feature Build

In [ ]:
class FeatureBuilder:
    """
    A class for feature extraction and transformation using various methods.

    Attributes:
        method (str): The feature extraction method ('tfidf', 'count', 'word2vec', 'bert', etc.).
        save_dir (str): Directory to save processed features.
        reduce_dim (str): Dimensionality reduction method ('pca', 'lda', or None).
        n_components (int): Number of components for dimensionality reduction.
        vectorizer (object): Vectorizer object for 'tfidf', 'count', or 'binary_count' methods.
        word2vec_model (object): Pretrained Word2Vec model.
        glove_model (object): Pretrained GloVe model.
        tokenizer (object): Tokenizer for BERT model.
        bert_model (object): BERT model for embedding extraction.
        reducer (object): Dimensionality reduction object (PCA or LDA).
    """

    def __init__(self, method="tfidf", save_dir="data/processed", reduce_dim=None, n_components=100):
        """
        Initializes the FeatureBuilder with a specified feature engineering method.

        Args:
            method (str): Feature engineering method ('tfidf', 'count', 'word2vec', 'bert', etc.).
            save_dir (str): Directory to save processed features.
            reduce_dim (str): Dimensionality reduction method ('pca', 'lda', or None).
            n_components (int): Number of components for dimensionality reduction.
        """
        self.method = method
        self.save_dir = save_dir
        self.reduce_dim = reduce_dim
        self.n_components = n_components
        self.reducer = None
        
        os.makedirs(save_dir, exist_ok=True)
        
        # Define models for vectorization
        if method == "tfidf":
            self.vectorizer = TfidfVectorizer(max_features=2000, stop_words="english")
        elif method == "count":
            self.vectorizer = CountVectorizer(max_features=2000)
        elif method == "binary_count":
            self.vectorizer = CountVectorizer(binary=True, max_features=2000)
        elif method == "word2vec":
            self.word2vec_model = api.load("word2vec-google-news-300")  # Pretrained Google News Word2Vec
        elif method == "glove":
            # self.glove_model = api.load("glove-wiki-gigaword-100")      # Pretrained GloVe embeddings
            self.glove_model = api.load("glove-wiki-gigaword-300")      # Pretrained GloVe embeddings
        elif method == "bert":
            self.tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
            self.bert_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
    
        # Initialize dimensionality reduction if required
        if self.reduce_dim == "pca":
            self.reducer = PCA(n_components=self.n_components)
        elif self.reduce_dim == "lda":
            self.reducer = LDA(n_components=self.n_components)

    def _apply_reducer(self, features, labels=None):
        """Applies dimensionality reduction if enabled."""
        if self.reducer is not None:
            if isinstance(self.reducer, LDA):
                assert labels is not None, "LDA requires class labels during transform."
                features = self.reducer.fit_transform(features, labels)
            else:
                features = self.reducer.fit_transform(features)
        return features
    
    def _get_word2vec_vector(self, doc):
        """
        Extracts the average Word2Vec embedding for a document.

        Args:
            doc (str): The document text.

        Returns:
            np.array: The averaged Word2Vec embedding.
        """
        tokens = doc.split()
        word_vectors = []
        for token in tokens:
            if token in self.word2vec_model: 
                word_vectors.append(self.word2vec_model[token])  # No need for '.wv'
        if word_vectors:
            return np.mean(word_vectors, axis=0)
        else:
            return np.zeros(self.word2vec_model.vector_size)

    def _get_glove_vector(self, doc):
        """
        Extracts the average GloVe embedding for a document.

        Args:
            doc (str): The document text.

        Returns:
            np.array: The averaged GloVe embedding.
        """
        tokens = doc.split()
        word_vectors = []
        for token in tokens:
            if token in self.glove_model:  
                word_vectors.append(self.glove_model[token])  # Use directly without '.wv'
        if word_vectors:
            return np.mean(word_vectors, axis=0)
        else:
            return np.zeros(self.glove_model.vector_size)

    def _get_bert_embedding(self, doc):
        """
        Extracts the BERT embedding for a document.

        Args:
            doc (str): The document text.

        Returns:
            np.array: The BERT embedding.
        """
        inputs = self.tokenizer(doc, return_tensors="pt", padding=True, truncation=True, max_length=512)
        with torch.no_grad():
            outputs = self.bert_model(**inputs)
        return outputs.pooler_output.squeeze(0).numpy()
    
    def fit(self, texts):
        """
        Fits the model to the text data by computing necessary statistics (e.g., vocabulary, embeddings).

        Args:
            texts (list): Raw text data.
            labels (list, optional): Class labels for LDA. Defaults to None.
        """
        if self.method in ["tfidf", "count", "binary_count"]:
            self.vectorizer.fit(texts)

        elif self.method in ["word2vec", "glove", "bert"]:
            pass
            
    def transform(self, texts, labels=None):
        """
        Transforms new data based on the fitted model.

        Args:
            texts (list): Raw text data.
            labels (list, optional): Class labels for LDA. Defaults to None.

        Returns:
            np.array: Transformed feature matrix.
        """
        if self.method in ["tfidf", "count", "binary_count"]:
            # Transform the new data using the fitted vectorizer
            features = self.vectorizer.transform(texts).toarray()
            # return self._apply_reducer(features, labels)

        elif self.method == "word2vec":
            # Use the pre-trained Word2Vec model to generate embeddings
            word2vec_embeddings = []
            for doc in tqdm(texts, desc="Processing Word2Vec", unit="document"):
                word2vec_embeddings.append(self._get_word2vec_vector(doc))
            features = np.array(word2vec_embeddings)
            # return features

        elif self.method == "glove":
            # Similar process for GloVe embeddings
            glove_embeddings = []
            for doc in tqdm(texts, desc="Processing GloVe", unit="document"):
                glove_embeddings.append(self._get_glove_vector(doc))
            features = np.array(glove_embeddings)
            # return features

        elif self.method == "bert":
            # Use the pre-trained BERT model to generate embeddings
            bert_embeddings = []
            for doc in tqdm(texts, desc="Processing BERT", unit="document"):
                bert_embeddings.append(self._get_bert_embedding(doc))
            features = np.array(bert_embeddings)
            # return features

        # Apply dimensionality reduction if applicable
        # return self._apply_reducer(features, labels)
        if self.reduce_dim == "lda":
            assert labels is not None, "LDA requires class labels (y)."
            # features = self.vectorizer.transform(texts).toarray()
            self.reducer.fit(features, labels)
            return self.reducer.transform(features)
        elif self.reduce_dim == "pca":
            # features = self.vectorizer.transform(texts).toarray()
            self.reducer.fit(features)
            return self.reducer.transform(features)
        else:
            return features

    def fit_transform(self, texts, labels=None):
        """
        Fits and transforms the text data by first fitting the model and then transforming it.

        Args:
            texts (list): Raw text data.

        Returns:
            np.array: Transformed feature matrix.
        """
        self.fit(texts)  # First fit the model (compute parameters)
        return self.transform(texts, labels if self.reduce_dim == "lda" else None)  # Then transform the data using the fitted model
    
    def _save_model(self):
        """
        Saves the fitted vectorizer/scaler for later use.
        """
        # Ensure the directory exists
        save_dir = self.save_dir if self.save_dir else "data/processed"
        os.makedirs(save_dir, exist_ok=True)  # Create directory if it doesn't exist  
        
        if self.method in ["tfidf", "count", "binary_count"]:
            file_path = os.path.join(self.save_dir, f"{self.method}_vectorizer.pkl")
            with open(file_path, "wb") as f:
                pickle.dump(self.vectorizer, f)
        elif self.method in ["word2vec", "glove"]:
            # Save the Word2Vec or GloVe model
            file_path = os.path.join(self.save_dir, f"{self.method}_model.pkl")
            with open(file_path, "wb") as f:
                if self.method == "word2vec":
                    pickle.dump(self.word2vec_model, f)
                elif self.method == "glove":
                    pickle.dump(self.glove_model, f)
        elif self.method == "bert":
            # Save the BERT tokenizer and model
            tokenizer_path = os.path.join(self.save_dir, "bert_tokenizer.pkl")
            model_path = os.path.join(self.save_dir, "bert_model.pkl")
            with open(tokenizer_path, "wb") as f:
                pickle.dump(self.tokenizer, f)
            with open(model_path, "wb") as f:
                pickle.dump(self.bert_model, f)
                
        if self.reducer is not None:
            reducer_path = os.path.join(self.save_dir, f"{self.reduce_dim}_reducer.pkl")
            with open(reducer_path, "wb") as f:
                pickle.dump(self.reducer, f)
    
    def _load_model(self):
        """
        Loads the previously saved vectorizer/scaler.
        """
        # Ensure the directory exists
        os.makedirs(self.save_dir, exist_ok=True)
        
        if self.method in ["tfidf", "count", "binary_count"]:
            file_path = os.path.join(self.save_dir, f"{self.method}_vectorizer.pkl")
            if not os.path.exists(file_path):
                raise FileNotFoundError(f"No saved model found at {file_path}. Run `fit_transform` first.")
            with open(file_path, "rb") as f:
                self.vectorizer = pickle.load(f)
        elif self.method in ["word2vec", "glove"]:
            file_path = os.path.join(self.save_dir, f"{self.method}_model.pkl")
            if not os.path.exists(file_path):
                raise FileNotFoundError(f"No saved model found at {file_path}. Run `fit_transform` first.")
            with open(file_path, "rb") as f:
                self.word2vec_model = pickle.load(f)
        elif self.method == "bert":
            tokenizer_path = os.path.join(self.save_dir, "bert_tokenizer.pkl")
            model_path = os.path.join(self.save_dir, "bert_model.pkl")
            if not os.path.exists(tokenizer_path) or not os.path.exists(model_path):
                raise FileNotFoundError(f"No saved BERT model found at {tokenizer_path} or {model_path}. Run `fit_transform` first.")
            with open(tokenizer_path, "rb") as f:
                self.tokenizer = pickle.load(f)
            with open(model_path, "rb") as f:
                self.bert_model = pickle.load(f)
        
        if self.reduce_dim:
            reducer_path = os.path.join(self.save_dir, f"{self.reduce_dim}_reducer.pkl")
            if not os.path.exists(reducer_path):
                raise FileNotFoundError(f"No saved reducer found at {reducer_path}.")
            with open(reducer_path, "rb") as f:
                self.reducer = pickle.load(f)

def build_vector_for_text(df_sampled, feature_methods, project_root, reduce_dim=None, n_components=50):
    """
    Builds feature vectors for text data using specified feature extraction methods.

    Args:
        df_sampled (pd.DataFrame): The sampled DataFrame containing text data.
        feature_methods (list): List of feature extraction methods to use.
        project_root (str): Root directory of the project.

    Returns:
        dict: Dictionary of training feature matrices for each method.
        dict: Dictionary of testing feature matrices for each method.
        pd.Series: Training labels.
        pd.Series: Testing labels.
    """
    X_train_features_dict = {}
    X_test_features_dict = {}

    # Step 1: First, split the DataFrame before feature extraction (to maintain X-y matching)
    df_train, df_test = train_test_split(df_sampled, test_size=0.2, random_state=42, stratify=df_sampled["target"])

    # Extract y_train and y_test **before feature extraction** to ensure data alignment
    y_train = df_train["target"].reset_index(drop=True)
    y_test = df_test["target"].reset_index(drop=True)

    logger.log_message("\n🔎 Running feature extraction...\n")
    for method in tqdm(feature_methods, desc="Feature Extraction Progress"):
        logger.log_message(f"\n🔍 Processing feature extraction using: {method}...")

        try:
            n_classes = len(y_train.unique())
            if reduce_dim == "lda":
                n_components = min(n_components, n_classes - 1)
                
            # Initialize FeatureBuilder for the current method
            # reduce_dim_method = reduce_dim if method in ["tfidf", "count", "binary_count"] else None
            reduce_dim_method = reduce_dim 

            feature_builder = FeatureBuilder(
                method=method,
                save_dir=os.path.join(project_root, "data", "processed"),
                reduce_dim=reduce_dim_method,  # Only apply reduction to vector-based methods
                n_components=n_components
            )

            # Step 2: Fit on training data ONLY
            feature_builder.fit(df_train["text_clean"].tolist())

            # Step 3: Transform train and test sets separately
            # X_train = feature_builder.transform(df_train["text_clean"].tolist(), n_classes if reduce_dim == "lda" else None)
            X_train = feature_builder.transform(df_train["text_clean"].tolist(), y_train.tolist() if reduce_dim == "lda" else None)
            # X_test = feature_builder.transform(df_test["text_clean"].tolist())
            X_test = feature_builder.transform(df_test["text_clean"].tolist(), y_test.tolist() if reduce_dim == "lda" else None)

            # Ensure feature matrices are DataFrames
            X_train_features_dict[method] = pd.DataFrame(X_train)
            X_test_features_dict[method] = pd.DataFrame(X_test)

            logger.log_message(f"✅ {method} - Train shape: {X_train.shape}, Test shape: {X_test.shape}")

        except Exception as e:
            logger.log_message(f"❌ Error with {method}: {e}. Skipping this method.")

    return X_train_features_dict, X_test_features_dict, y_train, y_test

# Models utils

In [ ]:

# --------------------------------------------------
# Loc defined
class BayesianNetworkClassifier(BaseEstimator, ClassifierMixin):
    """
    BayesianNetworkClassifier

    This module provides a custom Bayesian Network classifier implementing scikit-learn's BaseEstimator and ClassifierMixin.
    The classifier supports feature selection, discretization, dimensionality reduction, and inference using Bayesian Networks.

    Key functionalities:
    - Feature filtering based on unique values
    - Principal Component Analysis (PCA) for dimensionality reduction
    - Discretization of continuous features
    - Training a Bayesian Network using Maximum Likelihood Estimation
    - Inference using Belief Propagation or Variable Elimination
    - Compatibility with scikit-learn's API for easy integration
    """
    
    def __init__(self, structure=None, n_bins=2, strategy='kmeans', min_unique_values=2, max_features=20):
        """
        Initializes the BayesianNetworkClassifier with user-defined parameters.

        Args:
            structure (list, optional): A predefined structure for the Bayesian Network.
            n_bins (int, optional): Number of bins for discretization (default is 2).
            strategy (str, optional): Discretization strategy (default is 'kmeans').
            min_unique_values (int, optional): Minimum unique values required per feature (default is 2).
            max_features (int, optional): Maximum features allowed after PCA (default is 20).
        """
        self.structure = structure
        self.n_bins = n_bins
        self.strategy = strategy
        self.min_unique_values = min_unique_values
        self.max_features = max_features
        self.model = None
        self.inference = None
        self.feature_names = None
        self.discretizer = None
        self.pca = None
        self.filtered_columns = None  # Lưu cột sau khi lọc
    
    def fit(self, X, y):
        """
        Fits the Bayesian Network classifier to the training data.

        Steps:
        1. Converts `X` into a DataFrame if necessary.
        2. Filters out low-variance features with fewer than `min_unique_values` unique values.
        3. Applies PCA if the number of features exceeds `max_features`.
        4. Discretizes the features based on the selected strategy.
        5. Trains the Bayesian Network using Maximum Likelihood Estimation.
        6. Initializes the inference engine for predictions.

        Args:
            X (array-like or pd.DataFrame): Feature matrix.
            y (array-like): Target labels.

        Returns:
            self: Trained model instance.
        """
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=[f"col_{i}" for i in range(X.shape[1])])
        
        # Loại bỏ các đặc trưng không đủ đa dạng
        unique_counts = X.nunique()
        self.filtered_columns = unique_counts[unique_counts >= self.min_unique_values].index.tolist()
        if len(self.filtered_columns) < 1:
            raise ValueError(f"No features with at least {self.min_unique_values} unique values.")
        X = X[self.filtered_columns]
        
        logger.log_message(f"After filtering: {len(self.filtered_columns)} features remain.")
        logger.log_message(f"Unique values per feature before discretization:\n{X.nunique()}")
        
        # Giảm số đặc trưng bằng PCA
        if len(self.filtered_columns) > self.max_features:
            self.pca = PCA(n_components=self.max_features)
            X_reduced = self.pca.fit_transform(X)
            X = pd.DataFrame(X_reduced, columns=[f"feat_{i}" for i in range(self.max_features)])
            self.feature_names = X.columns.tolist()
            logger.log_message(f"After PCA: Reduced to {self.max_features} features.")
        else:
            self.feature_names = self.filtered_columns
        
        # Rời rạc hóa dữ liệu
        is_discrete = all(X[col].apply(lambda x: x.is_integer() if pd.notna(x) else True).all() for col in X.columns)
        if is_discrete:
            X_discrete = X.astype(int)
        else:
            if X.max().max() < 1e-5:
                X_discrete = (X > 0).astype(int)
            else:
                self.discretizer = KBinsDiscretizer(n_bins=self.n_bins, encode='ordinal', strategy=self.strategy)
                X_discrete = pd.DataFrame(self.discretizer.fit_transform(X), columns=self.feature_names)
        
        logger.log_message("Unique values per feature after discretization:\n", X_discrete.nunique())
        
        # Kiểm tra dữ liệu và nhãn
        unique_values_per_feature = X_discrete.nunique()
        if any(unique_values_per_feature < 2):
            raise ValueError("Some features have fewer than 2 unique values after processing.")
        y = pd.Series(y).astype(int).reset_index(drop=True)
        if y.nunique() < 2:
            raise ValueError(f"Label has fewer than 2 unique values in this fold: {y.value_counts().to_dict()}")
        
        # Tạo dữ liệu huấn luyện
        data = X_discrete.copy()
        data['label'] = y
        
        # Tạo cấu trúc đơn giản
        if self.structure is None:
            self.structure = [(feat, 'label') for feat in self.feature_names]
        
        # Huấn luyện mô hình
        self.model = BayesianNetwork(self.structure)
        self.model.fit(data, estimator=MaximumLikelihoodEstimator)
        # self.inference = VariableElimination(self.model)
        self.inference = BeliefPropagation(self.model)
        return self
    
    def predict(self, X):
        """
        Predicts class labels for the given test data.

        Steps:
        1. Converts `X` into a DataFrame if necessary.
        2. Applies the same filtering and transformations as in `fit()`.
        3. Performs inference using the Bayesian Network.
        
        Args:
            X (array-like or pd.DataFrame): Test feature matrix.

        Returns:
            np.array: Predicted class labels.
        """
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=[f"col_{i}" for i in range(X.shape[1])])
        
        # Lọc dữ liệu kiểm tra bằng các cột đã lọc trong fit()
        X = X[self.filtered_columns]
        
        # Áp dụng PCA nếu có
        if self.pca is not None:
            X = pd.DataFrame(self.pca.transform(X), columns=self.feature_names)
        
        # Áp dụng rời rạc hóa nếu có
        if self.discretizer is not None:
            X_discrete = pd.DataFrame(self.discretizer.transform(X), columns=self.feature_names)
        else:
            X_discrete = X.astype(int)
        
        # Dự đoán
        y_pred = []
        for i in range(len(X_discrete)):
            evidence = {k: v for k, v in X_discrete.iloc[i].to_dict().items() if pd.notna(v)}
            pred = self.inference.map_query(variables=['label'], evidence=evidence, show_progress=False)
            y_pred.append(pred['label'])
        return np.array(y_pred)
    
    def score(self, X, y):
        """
        Computes the accuracy of the classifier.

        Args:
            X (array-like or pd.DataFrame): Test feature matrix.
            y (array-like): True labels.

        Returns:
            float: Accuracy score.
        """
        y_pred = self.predict(X)
        return accuracy_score(y, y_pred)
    
    def get_params(self, deep=True):
        """
        Returns model parameters in a dictionary format.

        Args:
            deep (bool, optional): Whether to return parameters for sub-objects (default is True).

        Returns:
            dict: Model parameters.
        """
        return {"structure": self.structure, "n_bins": self.n_bins, "strategy": self.strategy, 
                "min_unique_values": self.min_unique_values, "max_features": self.max_features}
    
    def set_params(self, **params):
        """
        Sets model parameters dynamically.

        Args:
            **params: Keyword arguments containing parameter names and values.

        Returns:
            self: Model instance with updated parameters.
        """
        for param, value in params.items():
            setattr(self, param, value)
        return self

# --------------------------------------------------
# Hung defined
def create_population(num_features, population_size):
    """
    Creates an initial population of binary feature selectors.

    Args:
        num_features (int): Number of features.
        population_size (int): Size of the population.

    Returns:
        np.ndarray: Initial population of binary feature selectors.
    """
    return np.random.randint(2, size=(population_size, num_features))

def fitness_function(features, X_train, y_train):
    """
    Evaluates the fitness of a feature selection candidate.

    Args:
        features (np.ndarray): Binary feature selector.
        X_train (np.ndarray): Training feature matrix.
        y_train (np.ndarray): Training labels.

    Returns:
        float: Fitness score (cross-validation accuracy).
    """
    selected_features = [i for i, f in enumerate(features) if f == 1]
    if not selected_features:  # Avoid empty feature sets
        return 0

    X_train_selected = X_train[:, selected_features]

    nb_model = GaussianNB(var_smoothing=1e-8)
    try:
        scores = cross_val_score(nb_model, X_train_selected, y_train, cv=5)
        return np.mean(scores)
    except ValueError as e:
        logger.log_message(f"Error during cross-validation: {e}")
        return 0

def crossover(parent1, parent2):
    """
    Performs single-point crossover.

    Args:
        parent1 (np.ndarray): First parent.
        parent2 (np.ndarray): Second parent.

    Returns:
        tuple: Two offspring resulting from the crossover.
    """
    point = np.random.randint(1, len(parent1) - 1)
    offspring1 = np.concatenate((parent1[:point], parent2[point:]))
    offspring2 = np.concatenate((parent2[:point], parent1[point:]))
    return offspring1, offspring2

def mutate(individual, mutation_rate=0.1):
    """
    Mutates an individual with a given probability.

    Args:
        individual (np.ndarray): Individual to mutate.
        mutation_rate (float): Probability of mutation.

    Returns:
        np.ndarray: Mutated individual.
    """
    for i in range(len(individual)):
        if np.random.rand() < mutation_rate:
            individual[i] = 1 - individual[i]
    return individual

def genetic_algorithm(X_train, y_train, X_test, y_test, model_save_path=None, img_save_path=None, img_loss_path=None, population_size=20, num_generations=100, mutation_rate=0.1, crossover_rate=0.7):
    """
    Runs a genetic algorithm to optimize feature selection for Naive Bayes.

    Args:
        X_train (pd.DataFrame): Training feature matrix.
        y_train (pd.Series): Training labels.
        X_test (pd.DataFrame): Testing feature matrix.
        y_test (pd.Series): Testing labels.
        model_save_path (str, optional): Path to save the trained model. Defaults to None.
        population_size (int): Size of the population. Defaults to 20.
        num_generations (int): Number of generations. Defaults to 100.
        mutation_rate (float): Probability of mutation. Defaults to 0.1.
        crossover_rate (float): Probability of crossover. Defaults to 0.7.

    Returns:
        GaussianNB: Trained Naive Bayes model.
    """
    # Check if the model already exists
    if os.path.exists(model_save_path):
        logger.log_message(f"🔄 Loading existing model from {model_save_path}...")
        model_algorithm = joblib.load(model_save_path)
        return model_algorithm
    
    num_features = X_train.shape[1]
    population = create_population(num_features, population_size)

    for generation in range(num_generations):
        fitness_scores = [fitness_function(ind, X_train.values, y_train) for ind in population]
        
        # Normalize fitness scores to avoid division errors
        fitness_scores = np.array(fitness_scores)
        fitness_scores = np.clip(fitness_scores, 1e-5, None)
        
        probabilities = fitness_scores / np.sum(fitness_scores)

        # Select parents based on probabilities
        selected_indices = np.random.choice(np.arange(population_size), size=population_size, p=probabilities)
        selected_parents = [population[idx] for idx in selected_indices]

        next_generation = []
        for j in range(0, population_size, 2):
            if np.random.rand() < crossover_rate:
                offspring1, offspring2 = crossover(selected_parents[j], selected_parents[j + 1])
            else:
                offspring1, offspring2 = selected_parents[j], selected_parents[j + 1]
            
            next_generation.append(mutate(offspring1, mutation_rate))
            next_generation.append(mutate(offspring2, mutation_rate))

        population = next_generation  # Move to the next generation

    # Select the best individual
    best_individual = population[np.argmax(fitness_scores)]
    selected_features = [i for i, f in enumerate(best_individual) if f == 1]

    X_train_selected = X_train.iloc[:, selected_features]
    X_test_selected = X_test.iloc[:, selected_features]

    logger.log_message(f"Selected {len(selected_features)} features out of {num_features}")

    # Scaling the selected features
    scaler = MinMaxScaler()
    X_train_selected = scaler.fit_transform(X_train_selected)
    X_test_selected = scaler.transform(X_test_selected)

    # Train Naive Bayes with selected features
    nb_model = GaussianNB()
    
    # Set up K-Fold Cross-Validation
    logger.log_message("\n🎯 Running K-Fold Cross-Validation...")
    k_fold = KFold(n_splits=5, shuffle=True, random_state=42)

    # Prepare containers for metrics
    accuracy_scores = []
    roc_auc_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []
    training_losses = []
    validation_losses = []

    # Perform CV loop
    for train_index, val_index in tqdm(k_fold.split(X_train_selected), total=k_fold.get_n_splits(), desc="K-Fold Progress"):
        X_train_cv, X_val_cv = X_train_selected[train_index], X_train_selected[val_index]
        y_train_cv, y_val_cv = y_train[train_index], y_train[val_index]
        
        model_cv = GaussianNB()
        model_cv.fit(X_train_cv, y_train_cv)

        # Loss
        train_loss = get_training_loss(model_cv, X_train_cv, y_train_cv)
        val_loss = get_training_loss(model_cv, X_val_cv, y_val_cv)
        training_losses.append(train_loss)
        validation_losses.append(val_loss)

        # Predictions
        val_preds = model_cv.predict(X_val_cv)
        val_probs = model_cv.predict_proba(X_val_cv)[:, 1]

        # Metrics
        accuracy_scores.append(accuracy_score(y_val_cv, val_preds))
        roc_auc_scores.append(roc_auc_score(y_val_cv, val_probs))
        f1_scores.append(f1_score(y_val_cv, val_preds))
        precision_scores.append(precision_score(y_val_cv, val_preds))
        recall_scores.append(recall_score(y_val_cv, val_preds))

    # Print average scores
    logger.log_message(f'📊 Average Accuracy: {int(mean(accuracy_scores) * 100)}%')
    logger.log_message(f'📊 Average ROC AUC: {int(mean(roc_auc_scores) * 100)}%')
    logger.log_message(f'📊 Average F1 Score: {int(mean(f1_scores) * 100)}%')
    logger.log_message(f'📊 Average Precision: {int(mean(precision_scores) * 100)}%')
    logger.log_message(f'📊 Average Recall: {int(mean(recall_scores) * 100)}%')

    # Retrain final model on full training set
    nb_model.fit(X_train_selected, y_train)

    # Save model
    if model_save_path:
        joblib.dump(nb_model, model_save_path)
        logger.log_message(f'💾 Model saved to {model_save_path}')

    # Plot accuracy & ROC AUC per fold
    if img_save_path:
        plt.figure(figsize=(10, 6))
        plt.plot(range(1, len(accuracy_scores) + 1), accuracy_scores, label="Accuracy", marker='o')
        plt.plot(range(1, len(roc_auc_scores) + 1), roc_auc_scores, label="ROC AUC", marker='o')
        plt.title("Validation Performance Across K-Folds")
        plt.xlabel("Fold Number")
        plt.ylabel("Score")
        plt.legend()
        plt.savefig(img_save_path)
        plt.close()
        logger.log_message(f"📈 Plot saved to {img_save_path}")

    # Plot loss curves
    if img_loss_path:
        plt.figure(figsize=(10, 6))
        plt.plot(range(1, len(training_losses) + 1), training_losses, label="Training Loss", marker='o')
        plt.plot(range(1, len(validation_losses) + 1), validation_losses, label="Validation Loss", marker='o')
        plt.title("Training & Validation Loss Across K-Folds")
        plt.xlabel("Fold Number")
        plt.ylabel("Loss")
        plt.legend()
        plt.savefig(img_loss_path)
        plt.close()
        logger.log_message(f"📉 Loss plot saved to {img_loss_path}")

# --------------------------------------------------

def generate_binary_classification_model(X, y, model_algorithm, hyperparameters, needs_scaled = False, model_save_path="best_model.pkl", img_save_path=None, img_loss_path=None):
    """
    Generates everything required for training and validation of a binary classification model.

    Args:
        X (pd.DataFrame): Training features.
        y (pd.Series): Target values.
        model_algorithm (object): Model algorithm to train.
        hyperparameters (dict): Hyperparameters for tuning.
        needs_scaled (bool): Whether to scale the dataset. Defaults to False.
        model_save_path (str): Path to save the best model. Defaults to "best_model.pkl".
        img_save_path (str, optional): Path to save validation performance plot. Defaults to None.
        img_loss_path (str, optional): Path to save training loss plot. Defaults to None.

    Returns:
        object: Trained model.
    """
    # Check if the model already exists
    if os.path.exists(model_save_path):
        logger.log_message(f"🔄 Loading existing model from {model_save_path}...")
        model_algorithm = joblib.load(model_save_path)
        return model_algorithm
    
    logger.log_message(f"🚀 Training new model: {model_algorithm.__class__.__name__}...")
    # Performing a scaling on the data if required
    if needs_scaled:
        
        # Instantiating the StandardScaler
        scaler = StandardScaler()
        
        # Performing a fit_transform on the dataset
        scaled_features = scaler.fit_transform(X)
        
        # Transforming the StandardScaler output back into a Pandas DataFrame
        X = pd.DataFrame(scaled_features, index = X.index, columns = X.columns)
        
    # Instantiating a GridSearch object with the inputted model algorithm and hyperparameters
    gridsearchcv = GridSearchCV(estimator = model_algorithm,
                                param_grid = hyperparameters)
    
    # Fitting the training data to the GridSearch object
    gridsearchcv.fit(X, y)
    
    # Printing out the best hyperparameters
    logger.log_message(f'Best hyperparameters: {gridsearchcv.best_params_}')
    
    # Instantiating a new model object with the ideal hyperparameters from the GridSearch job
    model_algorithm.set_params(**gridsearchcv.best_params_)
    
    # Creating a container to hold each set of validation metrics
    accuracy_scores, roc_auc_scores, f1_scores, precision_scores, recall_scores = [], [], [], [], []
    training_losses, validation_losses = [], []
    
    # Instantiating the K-Fold cross validation object
    k_fold = KFold(n_splits = 5)
    
    logger.log_message("\n🎯 Running K-Fold Cross-Validation...")
    for train_index, val_index in tqdm(k_fold.split(X), total=k_fold.get_n_splits(), desc="K-Fold Progress"):

        # Splitting the training set from the validation set for this specific fold
        X_train, X_val = X.iloc[train_index, :], X.iloc[val_index, :]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]
        
        # Fitting the X_train and y_train datasets to the model algorithm
        model_algorithm.fit(X_train, y_train)
        
        # Compute losses
        train_loss = get_training_loss(model_algorithm, X_train, y_train)
        val_loss = get_training_loss(model_algorithm, X_val, y_val)

        training_losses.append(train_loss)
        validation_losses.append(val_loss)

        # Getting inferential predictions for the validation dataset
        val_preds = model_algorithm.predict(X_val)

        # Generating validation metrics by comparing the inferential predictions (val_preds) to the actuals (y_val)
        val_accuracy = accuracy_score(y_val, val_preds)
        val_roc_auc_score = roc_auc_score(y_val, val_preds)
        val_f1_score = f1_score(y_val, val_preds)
        val_precision_score = precision_score(y_val, val_preds)
        val_recall_score = recall_score(y_val, val_preds)
        
        # Appending the validation scores to the respective validation metric container
        accuracy_scores.append(val_accuracy)
        roc_auc_scores.append(val_roc_auc_score)
        f1_scores.append(val_f1_score)
        precision_scores.append(val_precision_score)
        recall_scores.append(val_recall_score)
        
    # Print average validation scores
    logger.log_message(f'📊 Average Accuracy: {int(mean(accuracy_scores) * 100)}%')
    logger.log_message(f'📊 Average ROC AUC: {int(mean(roc_auc_scores) * 100)}%')
    logger.log_message(f'📊 Average F1 Score: {int(mean(f1_scores) * 100)}%')
    logger.log_message(f'📊 Average Precision: {int(mean(precision_scores) * 100)}%')
    logger.log_message(f'📊 Average Recall: {int(mean(recall_scores) * 100)}%')
    
    # New added
    model_algorithm.fit(X, y)

    # Save the trained model
    joblib.dump(model_algorithm, model_save_path)
    logger.log_message(f'💾 Model saved to {model_save_path}')
    
    # If img_save_path is provided, plot the validation scores
    if img_save_path:
        plt.figure(figsize=(10, 6))

        # Plot Accuracy
        plt.plot(range(1, len(accuracy_scores) + 1), accuracy_scores, label="Accuracy", marker='o')

        # Plot ROC AUC
        plt.plot(range(1, len(roc_auc_scores) + 1), roc_auc_scores, label="ROC AUC", marker='o')

        # Add labels and title
        plt.title("Validation Performance Across K-Folds")
        plt.xlabel("Fold Number")
        plt.ylabel("Score")
        plt.legend()

        # Save the plot to the specified path
        plt.savefig(img_save_path)
        plt.close()
        logger.log_message(f"📈 Plot saved to {img_save_path}")
        
    # Plot loss curves
    if img_loss_path:
        plt.figure(figsize=(10, 6))
        plt.plot(range(1, len(training_losses) + 1), training_losses, label="Training Loss", marker='o')
        plt.plot(range(1, len(validation_losses) + 1), validation_losses, label="Validation Loss", marker='o')
        plt.title("Training & Validation Loss Across K-Folds")
        plt.xlabel("Fold Number")
        plt.ylabel("Loss")
        plt.legend()
        plt.savefig(img_loss_path)
        plt.close()
        logger.log_message(f"📉 Loss plot saved to {img_loss_path}")

    return model_algorithm

def get_training_loss(model, X_train, y_train):
    """
    Compute training loss based on model type.

    Args:
        model (object): Trained model.
        X_train (pd.DataFrame): Training feature matrix.
        y_train (pd.Series): Training labels.

    Returns:
        float: Training loss.
    """
    # Models that expose their loss during training
    if hasattr(model, "best_score_"):  # XGBoost
        return -model.best_score_

    if hasattr(model, "loss_"):  # Perceptron (Hinge loss)
        return model.loss_

    # Probabilistic models (e.g., HMM, Naive Bayes)
    if hasattr(model, "score"):  
        return -model.score(X_train, y_train)  # Negative log-likelihood

    # Support Vector Machines (hinge loss)
    if isinstance(model, SVC):
        y_pred = model.decision_function(X_train)
        return np.mean(np.maximum(0, 1 - y_train * y_pred))  # Hinge loss

    # Logistic Regression (log loss)
    if isinstance(model, LogisticRegression):
        y_proba = model.predict_proba(X_train)
        return log_loss(y_train, y_proba)

    # Decision Tree, Random Forest: No direct loss, use log loss
    if isinstance(model, (DecisionTreeClassifier, RandomForestClassifier)):
        y_proba = model.predict_proba(X_train)
        return log_loss(y_train, y_proba)

    return None  # Loss not available

# --------------------------------------------------
# Hung defined
def train_bayes_net(df, model_save_path):
    """
    Trains a Bayesian Network on the given DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame containing text data and target labels.
        model_save_path (str): Path to save the trained model.

    Returns:
        None
    """
    if os.path.exists(model_save_path):
        logger.log_message("✅ Model found! Loading...")
        # reader = BIFReader(model_save_path)
        # model = reader.get_model()
        logger.log_message("✅ Model loaded successfully!")
    else:  
        df_sampled = df
        
        vectorizer = CountVectorizer(binary=True, max_features=100) 
        X = vectorizer.fit_transform(df_sampled['text_clean']).toarray()
        y = df_sampled['target'].values
        
        # Chia dữ liệu thành tập huấn luyện và kiểm tra
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        # Chuyển đổi thành DataFrame để sử dụng với pgmpy
        feature_names = vectorizer.get_feature_names_out()
        train_df = pd.DataFrame(X_train, columns=feature_names)
        train_df['target'] = y_train
        
        # Xây dựng cấu trúc Bayesian Network
        # Giả sử mỗi từ phụ thuộc vào 'Sentiment'
        edges = [('target', word) for word in feature_names]
        model = BayesianNetwork(edges)
        
        # Học các bảng xác suất có điều kiện (CPT) từ dữ liệu
        model.fit(train_df, estimator=MaximumLikelihoodEstimator)

        # Suy luận và đánh giá mô hình
        inference = VariableElimination(model)
        
        # joblib.dump(model, model_save_path)
        # with open(model_save_path, "w") as f:
        #     f.write(model.to_bif())
        
        # logger.log_message(f'💾 Model saved to {model_save_path}')

        # Hàm dự đoán sentiment cho tập dữ liệu
        def predict_sentiment(model, inference, X, feature_names):
            predictions = []
            for i in range(X.shape[0]):
                evidence = {feature_names[j]: X[i, j] for j in range(len(feature_names))}
                result = inference.map_query(variables=['target'], evidence=evidence)
                predictions.append(result['target'])
            return np.array(predictions)

        def predict_sentiment_proba(model, inference, X, feature_names):
            proba_predictions = []
            for i in range(X.shape[0]):
                evidence = {feature_names[j]: X[i, j] for j in range(len(feature_names))}
                result = inference.query(variables=['target'], evidence=evidence)
                
                # Extract probability of target = 1 (assuming binary classification: 0 or 1)
                prob_1 = result.values[1]  # Probabilities are stored as an array, index 1 corresponds to class 1
                proba_predictions.append(prob_1)
            return np.array(proba_predictions)
        
        # Dự đoán trên tập kiểm tra
        y_pred = predict_sentiment(model, inference, X_test, feature_names)

        # Đánh giá độ chính xác
        # Compute metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='binary')
        recall = recall_score(y_test, y_pred, average='binary')
        f1 = f1_score(y_test, y_pred, average='binary')
        
        # ROC AUC can be computed if the model outputs probabilities
        # Handle models that do not support `predict_proba`
        if hasattr(model, "predict_proba"):
            logger.log_message("Has predict_proba")
            y_prob = model.predict_proba(X_test)[:, 1]  # Take the positive class probabilities
            roc_auc = roc_auc_score(y_test, y_prob)
        elif hasattr(model, "decision_function"):
            logger.log_message("Has decision_function")
            y_prob = model.decision_function(X_test)
            roc_auc = roc_auc_score(y_test, y_prob)
        else:
            logger.log_message("Does not have predict_proba or decision_function")
            y_proba = predict_sentiment_proba(model, inference, X_test, feature_names)
            roc_auc = roc_auc_score(y_test, y_proba)
            # roc_auc = "N/A"  # Not applicable for models like Perceptron

        # Print metrics
        logger.log_message("Model: Bayesian Network")
        logger.log_message("-" * 50)
        logger.log_message(f"Accuracy: {accuracy:.4f}")
        logger.log_message(f"Precision: {precision:.4f}")
        logger.log_message(f"Recall: {recall:.4f}")
        logger.log_message(f"F1 Score: {f1:.4f}")
        if hasattr(model, "predict_proba") or hasattr(model, "decision_function"):
            logger.log_message(f"ROC AUC: {roc_auc:.4f}")
        else:
            if roc_auc != "N/A":
                logger.log_message(f"ROC AUC: {roc_auc:.4f}")
            else:
                logger.log_message("ROC AUC: N/A")

def extract_features(text, word_features):
    """
    Extracts features from text based on a given vocabulary.

    Args:
        text (str): Input text.
        word_features (list): List of word features.

    Returns:
        np.ndarray: Array of feature indices.
    """
    words = text.split()  # Chuyển văn bản thành danh sách từ
    return np.array([word_features.index(word) for word in words if word in word_features])

def pad_sequence(seq, max_len):
    """
    Pads a sequence to a fixed length.

    Args:
        seq (np.ndarray): Input sequence.
        max_len (int): Maximum length for padding.

    Returns:
        np.ndarray: Padded sequence.
    """
    if len(seq) >= max_len:
        return seq[:max_len]
    return np.pad(seq, (0, max_len - len(seq)), mode='constant', constant_values=0)

def train_hmm(df, model_save_path):
    """
    Trains a Hidden Markov Model (HMM) on the given DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame containing text data and target labels.
        model_save_path (str): Path to save the trained model.

    Returns:
        None
    """
    df_sampled = df
    
    # Tạo tập từ vựng (chỉ lấy 3000 từ phổ biến nhất)
    all_words = nltk.FreqDist(word.lower() for text in df_sampled["text_clean"] for word in text.split())
    word_features = list(all_words.keys())[:5000]  # Lấy 3000 từ phổ biến nhất

    # Chuyển đổi dữ liệu text thành dạng số
    X = [extract_features(text, word_features) for text in df_sampled["text_clean"]]
    y = df_sampled["target"].values  # Nhãn (0: negative, 1: positive)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # HMM yêu cầu chuỗi có độ dài giống nhau -> Padding độ dài cố định (50 từ)
    max_len = 50
    
    X_train = np.array([pad_sequence(seq, max_len) for seq in X_train])
    X_test = np.array([pad_sequence(seq, max_len) for seq in X_test])

    # Huấn luyện HMM cho từng class (pos và neg)
    hmm = hmmlearn.hmm.GaussianHMM(n_components=2, covariance_type="diag", n_iter=100)
    hmm.fit(X_train)
    
    joblib.dump(hmm, model_save_path)
    logger.log_message(f'💾 Model saved to {model_save_path}')
    
    y_pred = hmm.predict(X_test)
    
    # Compute metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='binary')
    recall = recall_score(y_test, y_pred, average='binary')
    f1 = f1_score(y_test, y_pred, average='binary')
    
    # ROC AUC can be computed if the model outputs probabilities
    # Handle models that do not support `predict_proba`
    if hasattr(hmm, "predict_proba"):
        logger.log_message("Has predict_proba")
        y_prob = hmm.predict_proba(X_test)[:, 1]  # Take the positive class probabilities
        roc_auc = roc_auc_score(y_test, y_prob)
    elif hasattr(hmm, "decision_function"):
        logger.log_message("Has decision_function")
        y_prob = hmm.decision_function(X_test)
        roc_auc = roc_auc_score(y_test, y_prob)
    else:
        logger.log_message("Does not have predict_proba or decision_function")
        roc_auc = "N/A"  # Not applicable for models like Perceptron

    # Print metrics
    logger.log_message("Model: HMM")
    logger.log_message("-" * 50)
    logger.log_message(f"Accuracy: {accuracy:.4f}")
    logger.log_message(f"Precision: {precision:.4f}")
    logger.log_message(f"Recall: {recall:.4f}")
    logger.log_message(f"F1 Score: {f1:.4f}")
    if hasattr(hmm, "predict_proba") or hasattr(hmm, "decision_function"):
        logger.log_message(f"ROC AUC: {roc_auc:.4f}")
    else:
        logger.log_message("ROC AUC: N/A")
    
def train_graphical_model(df, model_name, model_save_path):
    """
    Trains a graphical model (HMM or Bayesian Network) on the given DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame containing text data and target labels.
        model_name (str): Name of the model to train ("hmm" or "bayesnet").
        model_save_path (str): Path to save the trained model.

    Returns:
        None
    """
    if model_name == "hmm":
        train_hmm(df, model_save_path)
    elif model_name == "bayesnet":
        train_bayes_net(df, model_save_path)

# --------------------------------------------------

def train_cnn_lstm(texts, labels, vocab_size=10000, max_length=500, embedding_dim=100, num_trials=5, epochs=30):
    """
    Trains a CNN-LSTM sentiment analysis model on given text data.

    Args:
        texts (list): List of sentences (raw text).
        labels (list): List of binary sentiment labels (0 for negative, 1 for positive).
        vocab_size (int): Size of vocabulary for tokenization. Defaults to 10000.
        max_length (int): Maximum sequence length for padding. Defaults to 500.
        embedding_dim (int): Dimension of the word embedding layer. Defaults to 100.
        num_trials (int): Number of hyperparameter tuning trials. Defaults to 5.
        epochs (int): Number of training epochs. Defaults to 10.

    Returns:
        keras.Sequential: Trained Keras model with the best hyperparameters.
        dict: Dictionary containing training and validation metrics.
    """
    # **Step 1: Text Preprocessing**
    tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    X_data = pad_sequences(sequences, maxlen=max_length, padding="pre")
    y_data = np.array(labels)  # Convert labels to NumPy array

    # **Step 2: Split Data for Training & Testing**
    X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

    # **Step 3: Build Model Function**
    def build_model(hp):
        model = keras.Sequential()

        # **Embedding Layer**
        model.add(layers.Embedding(
            input_dim=vocab_size, 
            output_dim=embedding_dim, 
            input_length=max_length
        ))

        # **CNN Block 1**
        model.add(layers.Conv1D(
            filters=hp.Int('filters_1', min_value=64, max_value=256, step=64),
            kernel_size=hp.Choice('kernel_size_1', values=[3, 5, 7]),
            activation="relu",
            padding="same"
        ))
        model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling1D(pool_size=2))

        # **CNN Block 2**
        model.add(layers.Conv1D(
            filters=hp.Int('filters_2', min_value=128, max_value=512, step=128),
            kernel_size=hp.Choice('kernel_size_2', values=[3, 5]),
            activation="relu",
            padding="same"
        ))
        model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling1D(pool_size=2))

        # **Bidirectional LSTM Layer**
        model.add(layers.Bidirectional(layers.LSTM(
            units=hp.Int('lstm_units', min_value=64, max_value=256, step=64),
            activation="tanh",
            return_sequences=False
        )))

        # **Fully Connected Layer**
        model.add(layers.Dense(
            units=hp.Int('dense_units', min_value=128, max_value=512, step=128),
            activation="relu"
        ))
        model.add(layers.Dropout(rate=hp.Float('dropout', min_value=0.3, max_value=0.6, step=0.1)))

        # **Output Layer**
        model.add(layers.Dense(1, activation="sigmoid"))

        # **Compile Model**
        model.compile(
            optimizer=keras.optimizers.Adam(hp.Choice('learning_rate', values=[5e-4, 1e-4, 5e-5, 1e-5])),
            loss="binary_crossentropy",
            metrics=["accuracy"]
        )

        return model

    # **Step 4: Initialize Keras Tuner**
    tuner = kt.RandomSearch(
        build_model,
        objective="val_accuracy",
        max_trials=num_trials,
        executions_per_trial=1,
        directory="tuner_results",
        project_name="cnn_lstm_tuning"
    )

    logger.log_message("\n🔍 Running Hyperparameter Tuning...")
    tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=32, validation_split=0.2, verbose=1)

    # **Step 5: Retrieve Best Model**
    best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
    best_model = tuner.hypermodel.build(best_hps)

    # **Step 6: Final Training with Best Model**
    logger.log_message("\n🚀 Training Final Model...")
    history = best_model.fit(X_train, y_train, epochs=epochs, validation_data=(X_test, y_test), batch_size=32, validation_split=0.2, verbose=1)
    
    # **Step 7: Predict on Validation Set**
    y_pred_prob = best_model.predict(X_test)
    y_pred = (y_pred_prob > 0.5).astype(int)

    # **Step 8: Compute Metrics**
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_prob)

    # **Step 9: Store Results**
    results = {
        "loss": history.history["loss"],
        "val_loss": history.history["val_loss"],
        "accuracy": history.history["accuracy"],
        "val_accuracy": history.history["val_accuracy"],
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "roc_auc": roc_auc
    }

    logger.log_message(f'🔹 loss: {history.history["loss"][-1]}')
    logger.log_message(f'🔹 val_loss: {history.history["val_loss"][-1]}')
    logger.log_message(f'🔹 accuracy: {history.history["accuracy"][-1]}')
    logger.log_message(f'🔹 val_accuracy: {history.history["val_accuracy"][-1]}')
    logger.log_message(f'🔹 precision: {precision}')
    logger.log_message(f'🔹 recall: {recall}')
    logger.log_message(f'🔹 f1_score: {f1}')
    logger.log_message(f'🔹 roc_auc: {roc_auc}')

    # **Step 10: Save the Best Model**
    best_model.save("best_cnn_lstm.keras")

    logger.log_message("\n✅ Model Training and Save Complete!")
    
    # === Plot Training & Validation Loss ===
    plt.figure(figsize=(10, 5))
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.title("📉 Training and Validation Loss over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("cnn_lstm_loss.png")
    plt.close()

    # === Plot Training & Validation Accuracy ===
    plt.figure(figsize=(10, 5))
    plt.plot(history.history["accuracy"], label="Training Accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
    plt.title("📈 Training and Validation Accuracy over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("cnn_lstm_accuracy.png")
    plt.close()

    logger.log_message("📊 Training curves saved: cnn_lstm_loss.png, cnn_lstm_accuracy.png")
    
    return best_model, results

# --------------------------------------------------

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
# from keras.preprocessing.text import Tokenizer
# from keras.utils import pad_sequences
import optuna
from torch.utils.data import Dataset, DataLoader
import json

START_TAG = "<START>"
STOP_TAG = "<STOP>"
tag_to_ix = {START_TAG: 0, STOP_TAG: 1, "NEG": 2, "POS": 3}
ix_to_tag = {v: k for k, v in tag_to_ix.items()}

def argmax(vec):
    return torch.argmax(vec)

def log_sum_exp(vec):
    max_score = vec.max()
    max_score_broadcast = max_score.view(1, -1).expand(1, vec.size()[1])
    return max_score + torch.log(torch.sum(torch.exp(vec - max_score_broadcast)))

class BiLSTM_CRF_FeatureExtractor(nn.Module):
    def __init__(self, vocab_size, tag_to_ix, embedding_dim, hidden_dim):
        super(BiLSTM_CRF_FeatureExtractor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim // 2,
                            num_layers=1, bidirectional=True, batch_first=True)
        self.hidden2tag = nn.Linear(hidden_dim, len(tag_to_ix))
        self.transitions = nn.Parameter(torch.randn(len(tag_to_ix), len(tag_to_ix)))
        self.tag_to_ix = tag_to_ix

        self.transitions.data[tag_to_ix[START_TAG], :] = -10000.
        self.transitions.data[:, tag_to_ix[STOP_TAG]] = -10000.

    def _get_lstm_features(self, sentence):
        embeds = self.embedding(sentence)
        lstm_out, _ = self.lstm(embeds)
        return self.hidden2tag(lstm_out)

    def _forward_alg(self, feats):
        init_alphas = torch.full((1, len(self.tag_to_ix)), -10000., device=feats.device)
        init_alphas[0][self.tag_to_ix[START_TAG]] = 0.
        forward_var = init_alphas
        for feat in feats:
            alphas_t = []
            for next_tag in range(len(self.tag_to_ix)):
                emit_score = feat[next_tag].view(1, -1).expand(1, len(self.tag_to_ix))
                trans_score = self.transitions[next_tag].view(1, -1)
                next_tag_var = forward_var + trans_score + emit_score
                alphas_t.append(log_sum_exp(next_tag_var).view(1))
            forward_var = torch.cat(alphas_t).view(1, -1)
        terminal_var = forward_var + self.transitions[self.tag_to_ix[STOP_TAG]]
        return log_sum_exp(terminal_var)

    def _score_sentence(self, feats, tags):
        score = torch.zeros(1, device=feats.device)
        tags = torch.cat([torch.tensor([self.tag_to_ix[START_TAG]], device=feats.device), tags])
        for i, feat in enumerate(feats):
            score += self.transitions[tags[i + 1], tags[i]] + feat[tags[i + 1]]
        score += self.transitions[self.tag_to_ix[STOP_TAG], tags[-1]]
        return score

    def _viterbi_decode(self, feats):
        backpointers = []
        init_vvars = torch.full((1, len(self.tag_to_ix)), -10000., device=feats.device)
        init_vvars[0][self.tag_to_ix[START_TAG]] = 0
        forward_var = init_vvars

        for feat in feats:
            bptrs_t = []
            viterbivars_t = []
            for next_tag in range(len(self.tag_to_ix)):
                next_tag_var = forward_var + self.transitions[next_tag]
                best_tag_id = argmax(next_tag_var)
                bptrs_t.append(best_tag_id.item())
                viterbivars_t.append(next_tag_var[0][best_tag_id].view(1))
            forward_var = (torch.cat(viterbivars_t) + feat).view(1, -1)
            backpointers.append(bptrs_t)

        terminal_var = forward_var + self.transitions[self.tag_to_ix[STOP_TAG]]
        best_tag_id = argmax(terminal_var)
        best_path = [best_tag_id.item()]
        for bptrs_t in reversed(backpointers):
            best_tag_id = bptrs_t[best_tag_id]
            best_path.append(best_tag_id)
        start = best_path.pop()
        best_path.reverse()
        return best_path

    def neg_log_likelihood(self, sentences, tags):
        feats = self._get_lstm_features(sentences)
        forward_score = self._forward_alg(feats[0])
        gold_score = self._score_sentence(feats[0], tags[0])
        return forward_score - gold_score

    def forward(self, sentences):
        feats = self._get_lstm_features(sentences)
        return self._viterbi_decode(feats[0])

class CRFSentimentDataset(Dataset):
    def __init__(self, input_ids, labels, max_len):
        self.input_ids = input_ids
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.input_ids[idx], dtype=torch.long)
        tag_label = "POS" if self.labels[idx] == 1 else "NEG"
        tag_id = tag_to_ix[tag_label]
        y = torch.tensor([tag_id] * self.max_len, dtype=torch.long)
        return x, y

def tag_sequence_to_sentiment(tag_seq):
    count_pos = tag_seq.count(tag_to_ix["POS"])
    count_neg = tag_seq.count(tag_to_ix["NEG"])
    return 1 if count_pos >= count_neg else 0

def train_crf_feature_extractor(texts, labels, vocab_size=10000, max_length=100, embedding_dim=100, num_trials=5, epochs=30):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.log_message(f"🖥️ Using device: {device}")

    tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    X_data = pad_sequences(sequences, maxlen=max_length, padding="post")
    y_data = np.array(labels)

    X_temp, X_test, y_temp, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

    def objective(trial):
        hidden_dim = trial.suggest_int("hidden_dim", 64, 256, step=64)
        lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)

        model = BiLSTM_CRF_FeatureExtractor(vocab_size, tag_to_ix, embedding_dim, hidden_dim).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        train_loader = DataLoader(CRFSentimentDataset(X_train, y_train, max_length), batch_size=32, shuffle=True)
        val_loader = DataLoader(CRFSentimentDataset(X_val, y_val, max_length), batch_size=32)

        model.train()
        for _ in range(3):
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                model.zero_grad()
                loss = model.neg_log_likelihood(x, y)
                loss.backward()
                optimizer.step()

        model.eval()
        y_pred, y_true = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                outputs = model(x)
                
                for i in range(x.size(0)):
                    decoded = model(x[i].unsqueeze(0))
                    y_pred.append(tag_sequence_to_sentiment(decoded))
                    y_true.append(1 if y[i][0].item() == tag_to_ix["POS"] else 0)

        return f1_score(y_true, y_pred)

    logger.log_message("🔍 Tuning hyperparameters...")
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=num_trials)
    best_params = study.best_params

    model = BiLSTM_CRF_FeatureExtractor(vocab_size, tag_to_ix, embedding_dim, best_params["hidden_dim"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=best_params["lr"])
    train_loader = DataLoader(CRFSentimentDataset(X_train, y_train, max_length), batch_size=32, shuffle=True)
    val_loader = DataLoader(CRFSentimentDataset(X_val, y_val, max_length), batch_size=32)

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            model.zero_grad()
            loss = model.neg_log_likelihood(x, y)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                total_val_loss += model.neg_log_likelihood(x, y).item()
        avg_val_loss = total_val_loss / len(val_loader)

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        logger.log_message(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

    test_loader = DataLoader(CRFSentimentDataset(X_test, y_test, max_length), batch_size=32)
    model.eval()
    y_pred, y_true = [], []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            outputs = model(x)
            for i in range(x.size(0)):
                decoded = model(x[i].unsqueeze(0))
                y_pred.append(tag_sequence_to_sentiment(decoded))
                y_true.append(1 if y[i][0].item() == tag_to_ix["POS"] else 0)


    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=["Negative", "Positive"])
    logger.log_message("\nClassification Report:\n", report)

    results = {
        "train_loss": history["train_loss"],
        "val_loss": history["val_loss"],
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "roc_auc": roc_auc
    }

    logger.log_message(f"\n✅ Test - Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, AUC: {roc_auc:.4f}")

    os.makedirs("crf_feature_model", exist_ok=True)
    torch.save(model.state_dict(), "best_crf_model.pt")
    with open("best_crf_model_config.json", "w") as f:
        json.dump(best_params, f, indent=4)

    plt.figure(figsize=(10, 5))
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Validation Loss")
    plt.title("📉 Train vs Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("crf_loss_curve.png")
    plt.close()

    return model, results


# --------------------------------------------------

def train_general_model(df, doc_lst, label_lst, model_name_lst, feature_methods, model_dict, param_dict, X_train_features_dict, X_test_features_dict, y_train, y_test):
    """
    Trains general models using specified feature extraction methods and model algorithms.

    Args:
        df (pd.DataFrame): Input DataFrame containing text data and target labels.
        doc_lst (list): List of documents (text data).
        label_lst (list): List of labels corresponding to the documents.
        model_name_lst (list): List of model names to train.
        feature_methods (list): List of feature extraction methods to use.
        model_dict (dict): Dictionary mapping model names to model classes.
        param_dict (dict): Dictionary mapping model names to hyperparameter grids.
        X_train_features_dict (dict): Dictionary of training feature matrices for each method.
        X_test_features_dict (dict): Dictionary of testing feature matrices for each method.
        y_train (pd.Series): Training labels.
        y_test (pd.Series): Testing labels.

    Returns:
        None
    """
    logger.log_message("\n🔎 Running feature extraction and model training loop...\n")
    
    for model_name in model_name_lst:
        logger.log_message(f"\n🚀 Training {model_name} models...\n")

        try:
            if model_name == "cnn" or model_name == "lstm":
                train_cnn_lstm(doc_lst, label_lst)
                
            elif model_name == "hmm" or model_name == "bayesnet":
                train_graphical_model(
                    df, 
                    model_name, 
                    model_save_path=f"best_{model_name}.pkl"
                )
                
            else:
                for method in feature_methods:
                    logger.log_message(f"🔎 Training with Method: {method}...")
                    
                    if model_name == "GA":
                        genetic_algorithm(
                            X_train_features_dict[method], 
                            y_train, 
                            X_test_features_dict[method], 
                            y_test, 
                            model_save_path=f"best_{model_name}_{method}.pkl",
                            img_save_path=f"best_{model_name}_{method}.png",
                            img_loss_path=f"best_{model_name}_{method}_loss.png"
                        )
                    
                    else:
                        model_api = model_dict[model_name]()
                        model_params = param_dict[model_name]
                        
                        generate_binary_classification_model(
                            X=X_train_features_dict[method], 
                            y=y_train, 
                            model_algorithm=model_api, 
                            hyperparameters=model_params, 
                            needs_scaled=False, 
                            model_save_path=f"best_{model_name}_{method}.pkl",
                            img_save_path=f"best_{model_name}_{method}.png",
                            img_loss_path=f"best_{model_name}_{method}_loss.png"
                        )
                        
        except Exception as e:
            logger.log_message(f"❌ Error with {model_name}: {e}")

# --------------------------------------------------  

def predict_general_model(model_names, feature_methods, X_test_features_dict, y_test, output_dir):
    """
    Predicts using trained models and evaluates their performance.

    Args:
        model_names (list): List of model names to use for prediction.
        feature_methods (list): List of feature extraction methods to use.
        X_test_features_dict (dict): Dictionary of testing feature matrices for each method.
        y_test (pd.Series): Testing labels.
        output_dir (str): Directory to save the prediction results.

    Returns:
        None
    """
    # Predict for each model
    for model_name in model_names:
        if model_name in ["GA", "hmm", "bayesnet", "lstm"]:
            logger.log_message(f"Already trained and tested model: {model_name}")
            continue
        for method in feature_methods:
            logger.log_message(f"🔎 Predicting with Model: {model_name}, Method: {method}...")
            
            try:
                if model_name in ["cnn"]:
                    # Load the saved deep learning model
                    model_filename = os.path.join(output_dir, f"best_{model_name}.keras")
                    model = tf.keras.models.load_model(model_filename)

                    # Retrieve and reshape features for CNN/LSTM
                    X_test_features = np.array(X_test_features_dict[method])
                    if model_name == "lstm":
                        input_shape = (1, X_test_features.shape[1])
                        X_test_features = X_test_features.reshape(X_test_features.shape[0], *input_shape)
                    else:
                        input_shape = (X_test_features.shape[1], 1)
                        X_test_features = X_test_features.reshape(-1, X_test_features.shape[1], 1)

                    # Make predictions
                    y_prob = model.predict(X_test_features).flatten()
                    y_pred = (y_prob > 0.5).astype(int)

                else:  # Handle Machine Learning models
                    # Load the saved model
                    model_filename = os.path.join(output_dir, f"best_{model_name}_{method}.pkl")
                    with open(model_filename, 'rb') as model_file:
                        model = joblib.load(model_file)

                    # Make predictions
                    y_pred = model.predict(X_test_features_dict[method])
                    
                    # ROC AUC can be computed if the model outputs probabilities
                    # Handle models that do not support `predict_proba`
                    if hasattr(model, "predict_proba"):
                        y_prob = model.predict_proba(X_test_features_dict[method])[:, 1]  # Take the positive class probabilities
                    elif hasattr(model, "decision_function"):
                        y_prob = model.decision_function(X_test_features_dict[method])
                    else:
                        y_prob = None

                # Compute metrics
                accuracy = accuracy_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred, average='binary')
                recall = recall_score(y_test, y_pred, average='binary')
                f1 = f1_score(y_test, y_pred, average='binary')
                roc_auc = roc_auc_score(y_test, y_prob) if y_prob is not None else "N/A"

                logger.log_message(f"Model: {model_name}")
                logger.log_message(f"Method: {method}")
                logger.log_message("-" * 50)
                logger.log_message(f"Accuracy: {accuracy:.4f}")
                logger.log_message(f"Precision: {precision:.4f}")
                logger.log_message(f"Recall: {recall:.4f}")
                logger.log_message(f"F1 Score: {f1:.4f}")
                logger.log_message(f"ROC AUC: {roc_auc if roc_auc != 'N/A' else 'N/A'}")
                # Print classification report for binary classification (0 = negative, 1 = positive)
                logger.log_message("\n🔬 Classification Report:")
                logger.log_message(classification_report(y_test, y_pred, labels=[0, 1], target_names=["negative", "positive"]))

                    
            except Exception as e:
                logger.log_message(f"❌ Error while predicting for {model_name} with {method}: {e}")

            
        logger.log_message("%" * 50)
        logger.log_message("%" * 50)

# --------------------------------------------------
# helper plot func
def plot_results(accuracy, roc_auc, train_loss, val_loss, img_save_path, img_loss_path):
    if img_save_path:
        plt.figure()
        plt.plot(accuracy, label="Accuracy", marker='o')
        plt.plot(roc_auc, label="ROC AUC", marker='o')
        plt.legend()
        plt.title("Validation Performance")
        plt.savefig(img_save_path)
        logger.log_message(f"📈 Performance plot saved to {img_save_path}")

    if img_loss_path:
        plt.figure()
        plt.plot(train_loss, label="Train Loss", marker='o')
        plt.plot(val_loss, label="Val Loss", marker='o')
        plt.legend()
        plt.title("Loss Curves")
        plt.savefig(img_loss_path)
        logger.log_message(f"📉 Loss plot saved to {img_loss_path}")


# Do here

## Dict

In [ ]:
# Dictionary for models
MODEL_DICT = {
    "decision_tree": DecisionTreeClassifier,
    "perceptron": Perceptron,
    "mlp": MLPClassifier,
    "bayesian": GaussianNB,
    "random_forest": RandomForestClassifier,
    "xgboost": xgb.XGBClassifier,
    "svm": SVC,
    "logistic_regression": LogisticRegression,
    "lda": LDA
} 

# Dictionary for model parameters
MODEL_PARAMS = {
    "lda": {
        "solver": ["lsqr", "eigen"],
        "shrinkage": [None, "auto"],  # Only used with 'lsqr' or 'eigen'
        "tol": [1e-4, 1e-3, 1e-2]     # Tolerance for convergence
    },
    
    "decision_tree": {
        "criterion": ["gini", "entropy"],
        "max_depth": [10, 20, 30, 40],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"]
    },
    
    "perceptron": {
        "max_iter": [1000, 2000],
        "tol": [1e-3, 1e-4],
        "eta0": [0.001, 0.01, 0.1],
        "penalty": [None, "l2", "l1"],
        "alpha": [0.0001, 0.001, 0.01]
    },
    
    "mlp": {
        "hidden_layer_sizes": [(100,)],
        "activation": ["tanh", "logistic"],
        "solver": ["sgd"],
        "alpha": [0.001, 0.01],
        "batch_size": [32],
        "max_iter": [1000],
    },
    
    "bayesian": {
        "priors": [None, [0.5, 0.5], [0.4, 0.6], [0.3, 0.7], [0.2, 0.8], [0.1, 0.9], [0.05, 0.95]],
        "var_smoothing": [1e-9, 1e-8, 1e-7]
    },
    
    "random_forest": {
        "n_estimators": [100],
        "max_depth": [10],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt", "log2"]
    },
    
    "xgboost": {
        "n_estimators": [100, 150],
        "learning_rate": [0.001, 0.01, 0.1],
        "max_depth": [10, 15]
    },
    
    "svm": {
        "kernel": ["linear"],
        "C": [0.1, 1],
        "gamma": ["scale", "auto"]
    },
    
    "logistic_regression": {
        "penalty": ["l1", "l2", "elasticnet", None],
        "C": [0.1, 1.0, 10.0],
        "max_iter": [1000, 2000]
    }
}

# Dictionary for dimensionality reduction methods
DIMENSIONALITY_REDUCTION_DICT = {
    "pca": PCA,
    "lda": LDA,
}

## Load dataset

In [ ]:
dataset_path = "/kaggle/input"
# df = pd.read_csv(f"{dataset_path}/final_clean_no_neutral_no_duplicates.csv")
df = pd.read_csv(f"{dataset_path}/final_clean_no_neutral_no_duplicates_v1.csv")

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df["target"] = df["target"].replace(4, 1)

In [ ]:
output_dir = "/kaggle/working"

In [ ]:
feature_methods = ["count"]
df_sampled = df.sample(n=1000, random_state=42)
# df_sampled = df

In [ ]:
doc_lst = df_sampled["text_clean"].tolist()
label_lst = df_sampled["target"].tolist()

In [ ]:
X_train_features_dict, X_test_features_dict, y_train, y_test = build_vector_for_text(df_sampled, feature_methods, output_dir)

In [ ]:
X_train_pca_dict, X_test_pca_dict, y_train, y_test = build_vector_for_text(df_sampled, feature_methods, output_dir, "pca", 100)

In [ ]:
model_name_lst = [
    # "decision_tree", # ok
    # "random_forest", # ok
    # "xgboost", 
    # "perceptron", # ok
    # "mlp", # lau but ok
    # "lstm",
    # "bayesian",
    # "GA",
    # "hmm",
    # "bayesnet",
    # "logistic_regression",
    # "svm",
    # "lda",
    "CRF"
]

# Train - Test

In [ ]:
train_general_model(df_sampled, doc_lst, label_lst, model_name_lst, feature_methods, MODEL_DICT, MODEL_PARAMS, X_train_features_dict, X_test_features_dict, y_train, y_test)

In [ ]:
predict_general_model(model_name_lst, feature_methods, X_test_features_dict, y_test, output_dir)

In [ ]:
train_general_model(df_sampled, doc_lst, label_lst, model_name_lst, feature_methods, MODEL_DICT, MODEL_PARAMS, X_train_pca_dict, X_test_pca_dict, y_train, y_test)

In [ ]:
predict_general_model(model_name_lst, feature_methods, X_test_pca_dict, y_test, output_dir)

# Remove output

In [ ]:
# import shutil
# import os

# # Path to the working directory
# working_dir = '/kaggle/working/'

# # Remove all files and subdirectories in the working directory
# for filename in os.listdir(working_dir):
#     file_path = os.path.join(working_dir, filename)
#     if os.path.isfile(file_path):
#         os.remove(file_path)
#     else:
#         shutil.rmtree(file_path)

# logger.log_message("All outputs cleared from Kaggle working directory.")


In [ ]:
# import os

# # Path to the working directory
# working_dir = '/kaggle/working/'

# # Loop through files in the working directory
# for filename in os.listdir(working_dir):
#     if filename.endswith(".zip"):  # Check if it's a zip file
#         file_path = os.path.join(working_dir, filename)
#         os.remove(file_path)  # Remove the zip file
#         logger.log_message(f"Removed: {file_path}")

# logger.log_message("✅ All zip files removed from Kaggle working directory.")


# Zip all

In [ ]:
# import shutil
# import os
# import zipfile

# # Define working directory and zip file path
# working_dir = "/kaggle/working/"
# zip_file_path = "/kaggle/working/output_files.zip"

# # Create a zip archive containing only specific files
# with zipfile.ZipFile(zip_file_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
#     for filename in os.listdir(working_dir):
#         if filename.endswith(".log") or filename.endswith(".png") or filename.endswith(".pkl"):
#             file_path = os.path.join(working_dir, filename)
#             zipf.write(file_path, filename)  # Save file with its name (without full path)

# logger.log_message(f"✅ Selected outputs zipped into: {zip_file_path}")